# Aula 12 — Geração e Limites

Como usar este notebook: a **Parte A** tem células que o professor roda e
explica durante a aula. Acompanhe na tela, sem precisar digitar nada. A
**Parte B** é com você: complete os exercícios nos lugares marcados com
`# SEU CODIGO AQUI`.

Nada aqui treina nada: o modelo já está pronto desde a Aula 11. Todas as
células rodam em segundos.

## Parte A: Demonstração

### Carregar o modelo e o tokenizador

É o mesmo código das aulas anteriores, junto num lugar só.

In [ ]:
import io
import json
import math
import re
import urllib.request

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn

URL_DADOS = "https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/"
# Alternativa para testar offline:
# URL_DADOS = "../../data/"

TAMANHO_VOCABULARIO = 1024
BLOCO = 128
DIMENSAO = 128
CABECAS = 4
CAMADAS = 4
OCULTA = 256


def baixar(nome, binario=False):
    if URL_DADOS.startswith("http"):
        dados = urllib.request.urlopen(URL_DADOS + nome).read()
        return dados if binario else dados.decode("utf-8")
    if binario:
        return open(URL_DADOS + nome, "rb").read()
    return open(URL_DADOS + nome, encoding="utf-8").read()


configuracao = json.loads(baixar("machado_bpe.json"))
PADRAO = re.compile(configuracao["padrao"])
fusoes = {(a, b): 256 + i for i, (a, b) in enumerate(configuracao["fusoes"])}

tabela = {i: bytes([i]) for i in range(256)}
for (a, b), novo in fusoes.items():
    tabela[novo] = tabela[a] + tabela[b]


def codificar(texto):
    saida = []
    for pedaco in PADRAO.findall(texto):
        simbolos = list(pedaco.encode("utf-8"))
        while len(simbolos) >= 2:
            candidatos = [p for p in zip(simbolos, simbolos[1:]) if p in fusoes]
            if not candidatos:
                break
            par = min(candidatos, key=lambda p: fusoes[p])
            novos, i = [], 0
            while i < len(simbolos):
                if i < len(simbolos) - 1 and (simbolos[i], simbolos[i + 1]) == par:
                    novos.append(fusoes[par])
                    i += 2
                else:
                    novos.append(simbolos[i])
                    i += 1
            simbolos = novos
        saida.extend(simbolos)
    return saida


def decodificar(ids):
    return b"".join(tabela[int(i)] for i in ids).decode("utf-8", errors="replace")


print(codificar("Não sei se a senhora"))

In [ ]:
def girar(x, base=10000.0):
    lote, cabecas, tokens, dim = x.shape
    metade = dim // 2
    frequencias = base ** (-torch.arange(0, metade).float() / metade)
    angulos = torch.arange(tokens).float()[:, None] * frequencias[None, :]
    cosseno, seno = angulos.cos()[None, None], angulos.sin()[None, None]
    primeira, segunda = x[..., :metade], x[..., metade:]
    return torch.cat([primeira * cosseno - segunda * seno,
                      primeira * seno + segunda * cosseno], dim=-1)


class Atencao(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(DIMENSAO, 3 * DIMENSAO, bias=False)
        self.saida = nn.Linear(DIMENSAO, DIMENSAO, bias=False)

    def forward(self, x):
        lote, tokens, dim = x.shape
        pergunta, chave, valor = self.qkv(x).split(DIMENSAO, dim=2)
        forma = (lote, tokens, CABECAS, dim // CABECAS)
        pergunta, chave, valor = [t.view(forma).transpose(1, 2)
                                  for t in (pergunta, chave, valor)]
        pergunta, chave = girar(pergunta), girar(chave)
        y = F.scaled_dot_product_attention(pergunta, chave, valor, is_causal=True)
        return self.saida(y.transpose(1, 2).reshape(lote, tokens, dim))


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.porta = nn.Linear(DIMENSAO, OCULTA, bias=False)
        self.subida = nn.Linear(DIMENSAO, OCULTA, bias=False)
        self.descida = nn.Linear(OCULTA, DIMENSAO, bias=False)

    def forward(self, x):
        return self.descida(F.silu(self.porta(x)) * self.subida(x))


class Bloco(nn.Module):
    def __init__(self):
        super().__init__()
        self.norma1 = nn.RMSNorm(DIMENSAO)
        self.norma2 = nn.RMSNorm(DIMENSAO)
        self.atencao = Atencao()
        self.mlp = MLP()

    def forward(self, x):
        x = x + self.atencao(self.norma1(x))
        return x + self.mlp(self.norma2(x))


class MiniLLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embutir = nn.Embedding(TAMANHO_VOCABULARIO, DIMENSAO)
        self.blocos = nn.ModuleList(Bloco() for _ in range(CAMADAS))
        self.norma = nn.RMSNorm(DIMENSAO)
        self.cabeca = nn.Linear(DIMENSAO, TAMANHO_VOCABULARIO, bias=False)
        self.cabeca.weight = self.embutir.weight

    def forward(self, indices):
        x = self.embutir(indices)
        for bloco in self.blocos:
            x = bloco(x)
        return self.cabeca(self.norma(x))


guardado = torch.load(io.BytesIO(baixar("mini_llm.pt", binario=True)),
                      weights_only=False)
modelo = MiniLLM()
modelo.load_state_dict(guardado["pesos"], strict=False)
modelo.eval()

print(f"modelo do passo {guardado['passo']}")
print(f"perda de validação: {guardado['perda_validacao']:.3f}")
print(f"perplexidade: {math.exp(guardado['perda_validacao']):.1f}")

### A saída do modelo é uma distribuição

Uma passada devolve 1.024 números, um por token. Nenhum deles é "a
resposta".

In [ ]:
@torch.no_grad()
def probabilidades_de(texto, temperatura=1.0):
    ids = codificar(texto)
    logits = modelo(torch.tensor([ids]))[0, -1]
    return F.softmax(logits / temperatura, dim=-1), logits


inicio = "Não sei se a senhora"
probabilidades, logits = probabilidades_de(inicio)
melhores = torch.topk(probabilidades, 10)

print(f'depois de "{inicio}":')
for indice, valor in zip(melhores.indices.tolist(), melhores.values.tolist()):
    print(f"  {decodificar([indice])!r:<10} {valor:>6.1%}")
print(f"  {'os outros 1014':<10} {1 - melhores.values.sum():>6.1%}")

### A temperatura

Divide todos os logits por `T` antes da softmax. Não muda a ordem dos
candidatos, só a distância entre eles.

$$P_j = \frac{e^{z_j / T}}{\sum_{k} e^{z_k / T}}$$

In [ ]:
oito = torch.topk(logits, 8).indices
rotulos = [decodificar([i]) for i in oito.tolist()]

plt.figure(figsize=(13, 4))
for posicao, temperatura in enumerate([0.3, 0.8, 1.5]):
    plt.subplot(1, 3, posicao + 1)
    p = F.softmax(logits / temperatura, dim=-1)[oito]
    plt.bar(rotulos, p.numpy(), color="#B85042")
    plt.xticks(rotation=45, ha="right")
    plt.ylim(0, 1)
    plt.title(f"T = {temperatura}")
plt.tight_layout()
plt.show()

### Cortar a cauda: top-k

Fora dos 40 melhores ainda sobra um terço da probabilidade, espalhada
por 984 candidatos ruins. O top-k joga todos fora.

In [ ]:
ordenadas = torch.sort(probabilidades, descending=True).values
for k in (10, 40, 100, 300):
    print(f"além do top-{k:>3}: {ordenadas[k:].sum():>6.1%} da probabilidade, "
          f"em {1024 - k} tokens")

### Gerar: repetir cinco passos

Passa pelo modelo, pega a última posição, aplica temperatura e corte,
sorteia, cola no fim, repete.

In [ ]:
@torch.no_grad()
def gerar(texto, quantos=80, temperatura=0.8, top_k=40, semente=11):
    torch.manual_seed(semente)
    indices = torch.tensor([codificar(texto)], dtype=torch.long)
    for _ in range(quantos):
        logits = modelo(indices[:, -BLOCO:])[:, -1, :]
        if temperatura == 0:
            proximo = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = logits / temperatura
            if top_k:
                corte = torch.topk(logits, top_k)[0][:, -1:]
                logits = logits.masked_fill(logits < corte, float("-inf"))
            proximo = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
        indices = torch.cat([indices, proximo], dim=1)
    return decodificar(indices[0].tolist())


for rotulo, temperatura, top_k in [
        ("sempre o mais provável", 0, 0),
        ("temperatura 0,3", 0.3, 40),
        ("temperatura 0,8 (padrão)", 0.8, 40),
        ("temperatura 1,5, sem corte", 1.5, 0)]:
    print(f"--- {rotulo} ---")
    print(gerar("A casa de", temperatura=temperatura, top_k=top_k))
    print()

### Ele copia ou inventa?

Procuramos no corpus o maior trecho que aparece igual no texto gerado.

In [ ]:
corpus = baixar("machado.txt")
gerado = gerar("A casa de", quantos=200, semente=21)
palavras = gerado.split()

maiores = []
for inicio_trecho in range(len(palavras)):
    tamanho = 0
    while inicio_trecho + tamanho < len(palavras):
        trecho = " ".join(palavras[inicio_trecho:inicio_trecho + tamanho + 1])
        if trecho not in corpus:
            break
        tamanho += 1
    maiores.append(tamanho)

maior = max(maiores)
onde = maiores.index(maior)
print(f"maior trecho copiado: {maior} palavras")
print(f"  {' '.join(palavras[onde:onde + maior])!r}")
print(f"média: {sum(maiores) / len(maiores):.2f} palavras")

## Parte B: Exercícios

Complete cada exercício no espaço marcado com `# SEU CODIGO AQUI`. Rode a
célula de verificação logo depois para conferir sua resposta.

### Exercício 1: a distribuição de uma frase sua

Rode e observe. Troque a frase e veja como as continuações mudam.

In [ ]:
minha_frase = "O menino olhou para a"
minhas_probabilidades, meus_logits = probabilidades_de(minha_frase)
minhas_melhores = torch.topk(minhas_probabilidades, 8)

print(f'depois de "{minha_frase}":')
for indice, valor in zip(minhas_melhores.indices.tolist(),
                         minhas_melhores.values.tolist()):
    print(f"  {decodificar([indice])!r:<10} {valor:>6.1%}")

In [ ]:
if abs(float(minhas_probabilidades.sum()) - 1.0) < 0.001:
    print("✅ Os 1.024 números somam 1: é uma distribuição de probabilidade.")
else:
    print("❌ Confira se você usou F.softmax.")

### Exercício 2: aplicando a temperatura

Calcule `probabilidades_frias` e `probabilidades_quentes` a partir de
`meus_logits`, com temperatura 0,3 e 2,0.

$$P_j = \text{softmax}\!\left(\frac{z_j}{T}\right)$$

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"maior probabilidade com T = 0,3: {probabilidades_frias.max():.1%}")
print(f"maior probabilidade com T = 1,0: {minhas_probabilidades.max():.1%}")
print(f"maior probabilidade com T = 2,0: {probabilidades_quentes.max():.1%}")

In [ ]:
if probabilidades_frias.max() > minhas_probabilidades.max() > probabilidades_quentes.max():
    print("✅ Temperatura baixa concentra, temperatura alta espalha.")
    print("   E o campeão continua o mesmo nos três: a ordem não muda.")
else:
    print("❌ Confira: dividir por 0,3 aumenta os logits, dividir por 2,0 diminui.")

### Exercício 3: o corte top-k

Complete `meu_top_k`, que devolve os logits com tudo fora dos `k`
melhores trocado por menos infinito.

Dica: `torch.topk(logits, k)[0][-1]` é o valor do k-ésimo melhor.

In [ ]:
def meu_top_k(logits, k):
    # SEU CODIGO AQUI
    return logits

In [ ]:
# SEU CODIGO AQUI

In [ ]:
cortados = meu_top_k(meus_logits, 5)
sobreviventes = torch.isfinite(cortados).sum()
print(f"tokens que sobraram: {sobreviventes}")
print(f"probabilidade deles: {F.softmax(cortados, dim=-1).sum():.0%}")

In [ ]:
if sobreviventes == 5:
    print("✅ Cinco tokens sobraram, e a softmax redistribuiu 100% entre eles.")
else:
    print(f"❌ Esperava 5 sobreviventes e vieram {sobreviventes}.")

### Exercício 4: o laço de geração

Complete `meu_gerador`. São cinco passos por token: passar pelo modelo,
pegar a última posição, aplicar temperatura e corte, sortear, colar.

In [ ]:
@torch.no_grad()
def meu_gerador(texto, quantos=60, temperatura=0.8, k=40):
    indices = torch.tensor([codificar(texto)], dtype=torch.long)
    for _ in range(quantos):
        pass  # SEU CODIGO AQUI
    return decodificar(indices[0].tolist())

In [ ]:
# SEU CODIGO AQUI

In [ ]:
torch.manual_seed(3)
meu_texto = meu_gerador("Era uma vez")
print(meu_texto)

In [ ]:
if len(codificar(meu_texto)) >= 60 and meu_texto.startswith("Era uma vez"):
    print("✅ Sessenta tokens novos, colados depois do começo que você deu.")
else:
    print("❌ Confira se você concatenou o token sorteado ao fim de indices.")

### Exercício 5: comparando temperaturas

Rode e observe o mesmo início com quatro temperaturas. Leia os quatro em
voz alta antes de continuar.

In [ ]:
for temperatura in (0.2, 0.5, 0.9, 1.6):
    torch.manual_seed(5)
    print(f"--- T = {temperatura} ---")
    print(meu_gerador("Era uma vez", quantos=50, temperatura=temperatura))
    print()

In [ ]:
print("Converse com um colega: em qual temperatura o texto começa a repetir,")
print("e em qual ele começa a inventar palavra que não existe?")

### Exercício 6: perplexidade de duas frases

Calcule a perplexidade que o modelo dá a cada frase de `frases`. Guarde
em `perplexidades`, na mesma ordem.

$$\text{perplexidade} = e^{\text{perda}}$$

Dica: use `F.cross_entropy(logits[:-1], ids[1:])`.

In [ ]:
frases = [
    "Não sei se a senhora quer alguma cousa.",   # parece Machado
    "O algoritmo processa os dados na nuvem.",   # não parece nada
]

In [ ]:
# SEU CODIGO AQUI

In [ ]:
for frase, valor in zip(frases, perplexidades):
    print(f"{valor:>7.1f}  {frase}")

In [ ]:
if perplexidades[0] < perplexidades[1]:
    print("✅ A frase que parece Machado surpreende menos o modelo.")
    print("   Perplexidade mede surpresa, não verdade nem qualidade.")
else:
    print("❌ Confira se você deslocou os alvos de uma casa.")

### Exercício 7: desafio, quanto ele copiou

Gere 150 tokens e descubra o maior trecho que aparece **igual** no
corpus. Guarde o tamanho em `maior_copia`.

Dica: para cada posição, vá aumentando o trecho enquanto ele continuar
aparecendo em `corpus`.

In [ ]:
torch.manual_seed(99)
meu_gerado = meu_gerador("A janela da", quantos=150)
minhas_palavras = meu_gerado.split()

In [ ]:
# SEU CODIGO AQUI

In [ ]:
posicao = tamanhos.index(maior_copia)
print(f"maior trecho copiado: {maior_copia} palavras")
print(f"  {' '.join(minhas_palavras[posicao:posicao + maior_copia])!r}")
print(f"média: {sum(tamanhos) / len(tamanhos):.2f} palavras")

In [ ]:
if maior_copia < 12:
    print("✅ Nada de longo foi copiado.")
    print("   O modelo guarda uma estatística sobre o texto, não o texto.")
else:
    print(f"Um trecho de {maior_copia} palavras é longo. Vale investigar:")
    print("costuma acontecer com frases muito repetidas no corpus.")

Agora, em texto: um colega diz que o modelo "sabe" quem foi Machado de
Assis. Explique em três frases por que essa palavra está errada, usando
o que você viu nesta aula. Edite esta célula (duplo clique nela) e
escreva sua resposta no lugar deste parágrafo.